# Total dijet systematic uncertainty

Consume existing CSV results; no source calculations are rerun. Each source is treated as **independent** and the symmetric fractional uncertainties are combined as $\delta_{total}=\sqrt{\sum_i\delta_i^2}$. Source notebooks already reduce Up/Down to a symmetric maximum or average. This notebook does not symmetrize, normalize, interpolate, or rebin.

Full means the unit-normalized CM dijet shape; fb means the unnormalized Forward/Backward ratio. All audit values remain fractions; plots alone convert to percent.

**Input contract:** defaults require all four sources and every canonical interval, and fail if a requested export is missing. An explicitly selected subset gives a total only for that named subset; availability is never used to infer a different configuration.

In [ ]:
# Initialize the environment using the same repository discovery as the other systematics notebooks.
%load_ext autoreload
%autoreload 2

from pathlib import Path
import os
import sys

# Jupyter may start the kernel in hist_analysis/notebooks rather than the root.
PROJECT_ROOT = next(
    (candidate for candidate in (Path.cwd(), *Path.cwd().parents)
     if (candidate / 'CMakeLists.txt').is_file()
     and (candidate / 'hist_analysis').is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError(
        'Cannot locate the jetAnalysis repository. Start Jupyter from its root.'
    )
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
from io import BytesIO
from IPython.display import Image, display
from hist_analysis.config.histograms import pt_bins
from hist_analysis.python.systematic_total import (
    combine_systematics, plot_systematics, archive_systematic_inputs,
    write_total_systematic_csv, total_systematic_configuration_tag,
    total_systematic_output_stem,
)

## Configuration and legacy metadata contract

`DIJET_BINNING` selects the canonical project bins. `SELECTED_PTAVE_BINS=None` requires every interval; an explicit subset must use canonical intervals. Configure full and F/B fit functions separately. The initial prescription is maximum / smoothed for every source; pointing uses full pol2 / F/B pol1 to match its exports. Pileup and pointing use a single absolute deviation, encoded as `systCombMax`.

CSV columns are headerless `(eta center, 1, eta half-width, fractional symmetric uncertainty)`. Filenames carry source/sample, full/fb, smoothing, selection, polynomial functions, fit weighting, combination, frame, acceptance, and ptAve. Pointing additionally carries the reference definition. JER and pointing use legacy directories without a binning suffix. JEU and pileup use normal/test subdirectories.

**Metadata declarations below are part of the input contract:** verify them against the producing run. Legacy data filenames omit beam direction; CSVs omit extraction mode, production identity, ratio-error settings, and smoothing origin. They cannot distinguish overwritten runs with identical filenames. Here `fit` declares the producing run used fits over the eta acceptance, and the CM smoothing origin was the source default `-0.465+0.00001`. If these facts differ, select separate input directories and update declarations. SHA256 hashes preserve exactly which bytes were consumed. These limitations cannot be reconstructed from the four CSV columns.

MC embedding and data are intentional source-specific inputs for uncertainties on the same pPb target measurement; they are recorded separately as `production_sample`. They must share direction, selection, acceptance, target sample, observable, and bins. This declaration does not claim the production samples are identical.
`POINTING_REFERENCE_DEFINITION` selects pointing CSVs: `refWithRecoPtAve` (default) requires the `RecoPtAve` filename tag; `refWithRefPtAve` requires `RefPtAve`. The chosen definition is recorded in provenance and total-output filenames. Missing requested files raise an error; the other definition is never substituted. Fit tags must match the source export (the pointing notebook currently uses full `pol2` and F/B `pol1`).

`OBSERVABLE` accepts `full`, `fb`, or `both` (default). `both` evaluates the two cases independently and saves each case's figures, audit/total CSVs, copied inputs, and provenance in separate `full/` and `fb/` subdirectories of the stable output directory.

Each source now selects `method=direct_ratio` or `folded_deviation`. Folded and raw bin-by-bin exports use explicit filename tags; direct-fit legacy files retain their names. The method is recorded in provenance. Direct-fit legacy names remain unchanged. `SYSTEMATIC_Y_RANGE` configures the common percent plot range (default 0–5%).

CMS headers follow the source notebooks: `CMS` for data/mixed inputs and `CMS Simulation` for MC only; `CMS_HEADER_LEFT` can override this explicitly. The default right label uses energy only, with no inferred luminosity or publication status. `SAVE_PNG` optionally adds PNGs next to the MPLHEP PDFs.


In [ ]:
DIJET_BINNING = 'normal'  # normal or test
OBSERVABLE = 'both'       # full, fb, or both (default), CM only
DIRECTION = 'combined'
SELECTION = 'jetId'
POINTING_REFERENCE_DEFINITION = 'refWithRecoPtAve'  # default; or refWithRefPtAve
POINTING_REFERENCE_TAGS = {
    'refWithRecoPtAve': 'RecoPtAve',
    'refWithRefPtAve': 'RefPtAve',
}
if POINTING_REFERENCE_DEFINITION not in POINTING_REFERENCE_TAGS:
    raise ValueError('POINTING_REFERENCE_DEFINITION must be refWithRecoPtAve or refWithRefPtAve')
ETA_CUT = 1.9
SELECTED_PTAVE_BINS = None  # None: every canonical interval; or explicit canonical tuples
OUTPUT_DIR = Path(os.environ.get('DIJET_TOTAL_SYSTEMATICS_OUTPUT_DIR',
    PROJECT_ROOT / 'hist_analysis/output/systematics_total')) / SELECTION / DIJET_BINNING
SYSTEMATIC_Y_RANGE = (0.0, 5.0)  # percent; all ptAve intervals; None for automatic limits
CMS_HEADER_LEFT = None  # infer CMS for data/mixed inputs, CMS Simulation for MC only
CMS_HEADER_RIGHT = 'pPb (8.16 TeV)'  # same energy-only header as source notebooks
SAVE_PNG = False
SAVE_OUTPUTS = True

# Variant choices are source-specific. None is allowed only for unused metadata.
systematics_config = {
    'JER': dict(enabled=True, method='folded_deviation', full_fit='pol2', fb_fit='pol1', combination='maximum',
                smoothing='smoothed', weights='Std', extraction='fit',
                directory=PROJECT_ROOT / 'hist_analysis/output/systematics_JER' / SELECTION,
                prefix='embedding_combined_jerSystematics', production_sample='embedding'),
    'JEU': dict(enabled=True, method='folded_deviation', full_fit='pol2', fb_fit='pol1', combination='maximum',
                smoothing='smoothed', weights='Std', extraction='fit',
                directory=PROJECT_ROOT / 'hist_analysis/output/systematics_JEU' / SELECTION / DIJET_BINNING,
                prefix='*_jeuSystematics', production_sample='data'),
    'pileup': dict(enabled=True, method='folded_deviation', full_fit='pol2', fb_fit='pol1', combination='maximum',
                smoothing='smoothed', weights='Std', extraction='fit',
                directory=PROJECT_ROOT / 'hist_analysis/output/systematics_pileup' / SELECTION / DIJET_BINNING,
                prefix='*_pileupSystematics', production_sample='data'),
    'pointing_resolution': dict(enabled=True, method='folded_deviation', full_fit='pol2', fb_fit='pol1', combination='maximum',
                smoothing='smoothed', weights='Std', extraction='fit',
                reference=POINTING_REFERENCE_DEFINITION,
                directory=PROJECT_ROOT / 'hist_analysis/output/systematics_pointing_resolution' / SELECTION,
                prefix='embedding_combined_pointing', production_sample='embedding'),
}

## Resolve the requested variants

A wildcard only permits the data trigger prefix to vary. Multiple trigger files for the same interval are ambiguous and rejected. No unavailable variant falls back to another fit order. `extraction` remains a declaration for legacy source filenames; each converted source now appends `_extraction_bin_by_bin` for raw exports. Polynomial tags remain present even for raw extraction. Existing source directories are retained.

To inspect a subset, explicitly set each source's `enabled` choice and `SELECTED_PTAVE_BINS`. Match every declared variant to the producing run; no source, interval, or fit is substituted automatically.

In [ ]:
if OBSERVABLE not in ('full', 'fb', 'both'):
    raise ValueError('OBSERVABLE must be full, fb, or both')
PTAVE_BINS = pt_bins('dijet', DIJET_BINNING) if SELECTED_PTAVE_BINS is None else SELECTED_PTAVE_BINS
OBSERVABLES = ('full', 'fb') if OBSERVABLE == 'both' else (OBSERVABLE,)
inputs_by_observable = {}
for observable in OBSERVABLES:
    inputs = {}
    for source, variant in systematics_config.items():
        if not variant['enabled']:
            continue
        if variant['combination'] not in ('maximum', 'average'):
            raise ValueError(f'{source}: choose maximum or average')
        if source in ('pileup', 'pointing_resolution') and variant['combination'] != 'maximum':
            raise ValueError(f'{source}: single-variation output uses systCombMax')
        if variant['extraction'] not in ('fit', 'bin_by_bin'):
            raise ValueError(f'{source}: unsupported extraction')
        if variant['smoothing'] not in ('smoothed', 'nonsmoothed') or variant['weights'] not in ('Std', '1'):
            raise ValueError(f'{source}: unsupported smoothing/weights')
        combination_tag = {'maximum': 'Max', 'average': 'Ave'}[variant['combination']]
        tag = (f"{SELECTION}_fullFit_{variant['full_fit']}_fbFit_{variant['fb_fit']}"
               f"_weights{variant['weights']}_systComb{combination_tag}")
        # Keep the reference tag before method/extraction, matching the producer.
        if source == 'pointing_resolution':
            if variant['reference'] not in POINTING_REFERENCE_TAGS:
                raise ValueError('pointing_resolution: reference must be refWithRecoPtAve or refWithRefPtAve')
            tag += '_' + POINTING_REFERENCE_TAGS[variant['reference']]
        method = variant['method']
        if method not in ('direct_ratio', 'folded_deviation'):
            raise ValueError(f'{source}: method must be direct_ratio or folded_deviation')
        if method != 'direct_ratio':
            tag += f'_method_{method}'
        if variant['extraction'] != 'fit':
            tag += f"_extraction_{variant['extraction']}"
        pattern = (f"{variant['prefix']}_systematic_{variant['smoothing']}_{observable}_relative_{tag}"
                   f"_etaCM_{round(10 * ETA_CUT):g}_ptave_{{low}}_{{high}}.csv")
        inputs[source] = dict(enabled=True, directory=variant['directory'], pattern=pattern,
            metadata=dict(observable=observable, frame='CM', direction=DIRECTION, selection=SELECTION,
                          eta_cut=ETA_CUT, units='fraction', convention='symmetric_relative',
                          target_sample='pPb 8.16 TeV dijets', binning=DIJET_BINNING,
                          production_sample=variant['production_sample'], extraction=variant['extraction'],
                          full_fit=variant['full_fit'], fb_fit=variant['fb_fit'],
                          combination=variant['combination'], smoothing=variant['smoothing'],
                          weights=variant['weights'], reference=variant.get('reference'),
                          method=variant['method'],
                          fit_range=((-ETA_CUT, ETA_CUT) if observable == 'full' else (0, ETA_CUT)), smoothing_origin=-0.465+0.00001))
    inputs_by_observable[observable] = inputs
    display(inputs)

In [ ]:
results_by_observable = {}
for observable, inputs in inputs_by_observable.items():
    audit_tables, provenance = combine_systematics(inputs, PTAVE_BINS, binning=DIJET_BINNING)
    display(provenance)
    # Independent scalar arithmetic checks every row, not only selected examples.
    import math
    sources = list(inputs)
    for interval, table in audit_tables.items():
        expected = np.array([math.fsum(float(row[s])**2 for s in sources)
                             for _, row in table.iterrows()])
        np.testing.assert_allclose(table.total.to_numpy()**2, expected, rtol=1e-14, atol=1e-18)
    print(f'{observable}: validated {len(audit_tables)} intervals; sources: {sources}')
    results_by_observable[observable] = dict(inputs=inputs, audit_tables=audit_tables,
                                            provenance=provenance, sources=sources)


## Audits, provenance, and figures

Each table contains the actual x-bin edges, one fractional column per enabled source, and their quadrature total. Saved filenames identify the selected subset; the adjacent manifest records exact paths, hashes, and all requested variant declarations. The stable output root is organized as `SELECTION/DIJET_BINNING/observable`, with MPLHEP figures under `mplhep/`. Repeating a configuration updates the same files; no timestamp directories are created.

Consumed source CSVs are copied unchanged into `individual_uncertainties/<source>/` with a stable configuration prefix; provenance records their original path, SHA256 and relative copied path. Each interval has its own `<stem>_provenance.json`, so a subset rerun does not replace another interval's record. Each total has an additional headerless `<stem>_total.csv` with `(eta center, 1, eta half-width, fractional total uncertainty)`, alongside the existing audit CSV and plot.

Filenames include direction, sources, selection, binning, observable, shared full/F/B fits, method and extraction (`mixed` when source-specific), acceptance and ptAve. A deterministic `cfg_<12 hex>` tag distinguishes all declared source variants and input-production directories, including pointing reference and weighting. It excludes CSV content hashes and selected interval lists, so refreshed inputs and subset reruns update the same logical outputs. Full declarations and consumed hashes remain in per-interval provenance.


In [ ]:
# Stable output paths: a rerun updates the same figures, audits and provenance.
for observable, result in results_by_observable.items():
    inputs = result['inputs']
    audit_tables = result['audit_tables']
    provenance = result['provenance']
    sources = result['sources']
    case_dir = OUTPUT_DIR / observable
    configuration_tag = total_systematic_configuration_tag(provenance)
    if SAVE_OUTPUTS:
        case_dir.mkdir(parents=True, exist_ok=True)
        (case_dir / 'mplhep').mkdir(parents=True, exist_ok=True)
        provenance = archive_systematic_inputs(provenance, case_dir,
            filename_prefix=configuration_tag + '_')
    for interval, table in audit_tables.items():
        print(f'{observable}, ptAve {interval}, fractions; included sources: {sources}')
        display(table)
        fig = plot_systematics(table, provenance, interval, y_range=SYSTEMATIC_Y_RANGE,
            cms_header_left=CMS_HEADER_LEFT, cms_header_right=CMS_HEADER_RIGHT)
        # Explicit PNG preview, as in the source-notebook CMS renderer.
        preview = BytesIO()
        fig.savefig(preview, format='png', dpi=100)
        display(Image(data=preview.getvalue()))
        if SAVE_OUTPUTS:
            stem = total_systematic_output_stem(provenance, interval)
            table.to_csv(case_dir / f'{stem}.csv', index=False)
            write_total_systematic_csv(table, case_dir / f'{stem}_total.csv')
            interval_provenance = provenance[
                (provenance.pt_low == interval[0]) & (provenance.pt_high == interval[1])]
            interval_provenance.to_json(case_dir / f'{stem}_provenance.json', orient='records', indent=2)
            fig.savefig(case_dir / 'mplhep' / f'{stem}.pdf')
            if SAVE_PNG:
                fig.savefig(case_dir / 'mplhep' / f'{stem}.png', dpi=160)
        import matplotlib.pyplot as plt
        plt.close(fig)
